# Bronze: Store — Raw File Ingestion

Loads raw `store` CSV files from the source volume into the
bronze table **`saleslake_{env}.bronze_{env}.rawStore`** using
Databricks `COPY INTO` for idempotent, exactly-once ingestion.

In [0]:
dbutils.widgets.dropdown(name="environment", defaultValue="dev",
                         choices=["dev","qa","prd"], label="select Environment")
env = dbutils.widgets.get("environment")
print(f"Environment: {env}")


In [0]:
tablName    = f"saleslake_{env}.bronze_{env}.rawStore"
srcFileLoc  = f"s3://saleslake-197180824086-eu-north-1-an/saleslake/{env}/source_files/store/"
print(f"Target table : {tablName}")
print(f"Source files : {srcFileLoc}")


In [0]:
spark.sql(f"""
COPY INTO {tablName}
FROM (
    SELECT store_id, store_code, store_name, store_type, address, city, state, region_id, manager_name, opening_date, square_feet, status,
           current_timestamp() AS ingest_ts
    FROM "{srcFileLoc}"
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS  ('mergeSchema' = 'false')
""")

print(f"Bronze load complete for {tablName}")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {tablName}").display()
